<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/%5B03A%5D-Build_Screening_Worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3: Build Screening Worksheet

This notebook builds the manual screening worksheet from the raw EDGAR
candidate pool, plus a 20% random subsample set aside for independent
re-coding.

The Cohen's kappa reliability check that used to be Part B of this notebook
now lives in its own file, `03g_kappa_check.ipynb`, run later in the
sequence (after `03f`) - so this file only ever does one job: build the
worksheet.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "your_email@example.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 223 (delta 90), reused 165 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 1.48 MiB | 15.77 MiB/s, done.
Resolving deltas: 100% (90/90), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

## Build the worksheet

Converts the raw EDGAR candidate pool into a screening worksheet with blank
columns for manual review, plus a 20% random subsample set aside for
independent re-coding.

In [ ]:
import pandas as pd


def build_worksheet():
    src = os.path.join(RAW_DIR, "edgar_candidate_events.csv")
    df = pd.read_csv(src)

    df["file_date"] = pd.to_datetime(df["file_date"])
    df = df.sort_values("file_date").drop_duplicates(subset=["cik", "file_date"], keep="first")

    df["is_genuine_ai_event"] = ""       # Y/N - excludes incidental AI mentions
    df["announcement_type"] = ""          # partnership / R&D / M&A
    df["confounding_event_flag"] = ""     # Y/N - other material event in [-2,+2]
    df["trading_halt_flag"] = ""          # Y/N
    df["sufficient_history_flag"] = ""    # Y/N - >=120 trading days pre-event
    df["exclude_reason"] = ""             # free text if excluded
    df["filing_url"] = df.apply(
        lambda r: f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={r['cik']}",
        axis=1,
    )
    df["screener_notes"] = ""

    df.to_csv(SCREENING_FILE, index=False)
    print(f"Worksheet built: {len(df)} candidate events -> {SCREENING_FILE}")

    n_subsample = max(1, int(len(df) * 0.20))
    subsample = df.sample(n=n_subsample, random_state=42)
    subsample_out = subsample[["accession_no", "company_name", "file_date"]].copy()
    subsample_out["recoder_announcement_type"] = ""
    subsample_out["recoder_is_genuine_ai_event"] = ""
    subsample_out.to_csv(RECODE_FILE, index=False)
    print(f"20% re-coding sample ({n_subsample} events) -> {RECODE_FILE}")
    return df


worksheet = build_worksheet()
worksheet.head()

Worksheet built: 7744 candidate events -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv
20% re-coding sample (1548 events) -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_recode_sample.csv


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes
1058,"""AI-powered""",1013857,PEGASYSTEMS INC (PEGA) (CIK 0001013857),8-K,2023-01-03,0001193125-23-000843:d442682dex991.htm,0001193125-23-000843,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
4970,"""AI capabilities""",876167,PROGRESS SOFTWARE CORP /MA (PRGS) (CIK 00008...,8-K,2023-01-03,0000876167-23-000004:pressrelease-marklogic.htm,0000876167-23-000004,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
1712,"""AI-powered""",1293818,OPGEN INC (OPGN) (CIK 0001293818),8-K,2023-01-04,0001079973-23-000010:ex99x1.htm,0001079973-23-000010,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
8248,"""neural network""",278165,OMNIQ Corp. (OMQS) (CIK 0000278165),8-K,2023-01-04,0001493152-23-000229:ex99-1.htm,0001493152-23-000229,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
7953,"""deep learning""",1577445,ScoutCam Inc. (ODYS) (CIK 0001577445),8-K,2023-01-05,0001493152-23-000438:ex99-1.htm,0001493152-23-000438,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} add "data/raw/screening_recode_sample.csv"
!git -C {BASE_DIR} commit -m "Step 3: build screening worksheet + recode sample"
!git -C {BASE_DIR} push

[main a6c9791] Step 3: build screening worksheet + recode sample
 2 files changed, 9294 insertions(+), 3122 deletions(-)
 rewrite data/raw/screening_recode_sample.csv (96%)
 rewrite data/raw/screening_worksheet.csv (89%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 341.35 KiB | 3.75 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   3db3fed..a6c9791  main -> main


## Next in the sequence

Continue to `03b_cik_ticker_map.ipynb`, then `03c`, `03d`, `03e`. The
manual reading step happens after `03e`. Once your manual review is
complete and merged via `03f`, and your independent re-coder's file is
also complete, move on to `03g_kappa_check.ipynb`.